Take our chosen final model, LightGBM, retrain it on all historical data available, then forecast dates after the dataset ends under different promotion plans.

In [ ]:
# =========================================================
# Phase 6 — Promotion-Aware Forecasting
# =========================================================

import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor

In [ ]:
# Load the prepared continuous daily series.

data = pd.read_csv(
    "../data/favorita/store3_item589403_tft_ready.csv"
)

# Convert date text into datetime.
data["date"] = pd.to_datetime(data["date"])

print("Rows:", len(data))
print(
    "Historical period:",
    data["date"].min(),
    "to",
    data["date"].max()
)

In [ ]:
# =========================================================
# Create LightGBM historical features
# =========================================================

# Lag features:
# each row receives the sales value from N days earlier.
data["lag_1"] = data["unit_sales"].shift(1)
data["lag_7"] = data["unit_sales"].shift(7)
data["lag_14"] = data["unit_sales"].shift(14)
data["lag_28"] = data["unit_sales"].shift(28)
data["lag_56"] = data["unit_sales"].shift(56)


# Rolling-average features must use only PAST sales.
# shift(1) ensures today's target is not included.
historical_sales = data["unit_sales"].shift(1)

data["rolling_mean_7"] = (
    historical_sales.rolling(7).mean()
)

data["rolling_mean_14"] = (
    historical_sales.rolling(14).mean()
)

data["rolling_mean_28"] = (
    historical_sales.rolling(28).mean()
)

data["rolling_mean_56"] = (
    historical_sales.rolling(56).mean()
)


# Define the exact feature set selected in Phase 5.
feature_columns = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "lag_56",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_mean_56",
    "time_idx",
    "day_of_week",
    "month",
    "promotion_status",
]

target_column = "unit_sales"


# Remove the first rows that do not yet have enough
# historical context for the 56-day features.
final_training_df = data.dropna(
    subset=feature_columns + [target_column]
).copy()


print("Rows available for final LightGBM training:",
      len(final_training_df))

print(
    "Final training period:",
    final_training_df["date"].min(),
    "to",
    final_training_df["date"].max()
)

In [ ]:
# =========================================================
# Prepare final LightGBM training data
# =========================================================

X_final = final_training_df[
    feature_columns
].copy()

y_final = final_training_df[
    target_column
].copy()


# Tell LightGBM which columns are categorical.
categorical_columns = [
    "day_of_week",
    "month",
    "promotion_status",
]

for column in categorical_columns:
    X_final[column] = X_final[column].astype("category")


print("X_final shape:", X_final.shape)
print("y_final shape:", y_final.shape)

In [ ]:
# =========================================================
# Train the final production LightGBM model
# =========================================================

final_lightgbm_model = LGBMRegressor(
    # Use a reasonable number of trees based on what we learned
    # during Phase 5 backtesting.
    n_estimators=200,

    # Small learning rate so each tree makes a controlled correction.
    learning_rate=0.03,

    # Maximum number of terminal leaves per tree.
    num_leaves=31,

    # Keep training reproducible.
    random_state=42
)


# Train on ALL available historical rows.
final_lightgbm_model.fit(
    X_final,
    y_final
)


print("Final production LightGBM trained successfully.")
print("Training rows used:", len(X_final))

In [ ]:
# =========================================================
# Create the future forecasting calendar
# =========================================================

# Last date for which we have real historical sales.
last_historical_date = data["date"].max()

# Start forecasting from the NEXT day.
forecast_start_date = (
    last_historical_date
    + pd.Timedelta(days=1)
)

# For now, create a 14-day forecast horizon.
forecast_horizon = 14

future_dates = pd.date_range(
    start=forecast_start_date,
    periods=forecast_horizon,
    freq="D"
)

# Create a DataFrame containing those future dates.
future_df = pd.DataFrame({
    "date": future_dates
})

print("Last historical date:", last_historical_date)
print("Forecast starts:", forecast_start_date)
print("Forecast horizon:", forecast_horizon, "days")

future_df

In [ ]:
# =========================================================
# Create future promotion scenarios
# =========================================================

# Scenario A:
# Assume there are NO promotions during the next 14 days.
no_promotion_scenario = future_df.copy()

no_promotion_scenario["promotion_status"] = (
    "no_promotion"
)


# Scenario B:
# The manager/user plans promotions on these future dates.
planned_promotion_dates = [
    pd.Timestamp("2017-08-20"),
    pd.Timestamp("2017-08-21"),
    pd.Timestamp("2017-08-27"),
]


# Start Scenario B as no-promotion everywhere.
promotion_scenario = future_df.copy()

promotion_scenario["promotion_status"] = (
    "no_promotion"
)


# Change only the dates selected by the user
# from "no_promotion" to "promotion".
promotion_scenario.loc[
    promotion_scenario["date"].isin(
        planned_promotion_dates
    ),
    "promotion_status"
] = "promotion"


print("Scenario A — No promotions:")
display(no_promotion_scenario)

print("\nScenario B — Planned promotions:")
display(promotion_scenario)

In [ ]:
# =========================================================
# Add known future calendar features
# =========================================================

# The historical time_idx ends at 1686.
# Therefore, the first future day should start at 1687.
last_time_idx = data["time_idx"].max()


def add_future_calendar_features(scenario_df):
    # Work on a copy so we don't accidentally modify
    # the original scenario DataFrame.
    scenario_df = scenario_df.copy()

    # Continue time_idx directly after the historical data.
    scenario_df["time_idx"] = range(
        int(last_time_idx) + 1,
        int(last_time_idx) + 1 + len(scenario_df)
    )

    # These calendar values are known in advance.
    scenario_df["day_of_week"] = (
        scenario_df["date"].dt.day_name()
    )

    scenario_df["month"] = (
        scenario_df["date"].dt.month.astype(str)
    )

    return scenario_df


# Add calendar features to both scenarios.
no_promotion_scenario = add_future_calendar_features(
    no_promotion_scenario
)

promotion_scenario = add_future_calendar_features(
    promotion_scenario
)


print("Scenario A:")
display(no_promotion_scenario)

print("\nScenario B:")
display(promotion_scenario)

In [ ]:
# =========================================================
# Prepare historical sales for recursive forecasting
# =========================================================

# Convert all historical unit_sales values into a Python list.
#
# Example:
# [5, 7, 6, 8, ...]
#
# During forecasting, we will append each new prediction
# to this list so that it becomes available as "history"
# for predicting the following future day.
historical_sales_values = data["unit_sales"].tolist()


print("Number of historical days:",
      len(historical_sales_values))

print(
    "Last 10 historical sales values:",
    historical_sales_values[-10:]
)

In [ ]:
# =========================================================
# Recursive LightGBM forecasting function
# =========================================================

def recursive_forecast(model, scenario_df, historical_sales):
    
    # Make a copy because we will keep adding predictions
    # to this list as we move through the future dates.
    sales_history = historical_sales.copy()

    # Store each future prediction here.
    predictions = []

    # Go through the future dates ONE DAY AT A TIME.
    for _, row in scenario_df.iterrows():

        # -------------------------------------------------
        # 1. Create lag features
        # -------------------------------------------------
        # [-1] = yesterday
        # [-7] = 7 days ago
        # and so on.
        lag_1 = sales_history[-1]
        lag_7 = sales_history[-7]
        lag_14 = sales_history[-14]
        lag_28 = sales_history[-28]
        lag_56 = sales_history[-56]


        # -------------------------------------------------
        # 2. Create rolling-average features
        # -------------------------------------------------
        # Take the mean of the most recent N sales values.
        rolling_mean_7 = np.mean(sales_history[-7:])
        rolling_mean_14 = np.mean(sales_history[-14:])
        rolling_mean_28 = np.mean(sales_history[-28:])
        rolling_mean_56 = np.mean(sales_history[-56:])


        # -------------------------------------------------
        # 3. Build ONE row containing the same 13 features
        #    that LightGBM saw during training.
        # -------------------------------------------------
        future_features = pd.DataFrame({
            "lag_1": [lag_1],
            "lag_7": [lag_7],
            "lag_14": [lag_14],
            "lag_28": [lag_28],
            "lag_56": [lag_56],

            "rolling_mean_7": [rolling_mean_7],
            "rolling_mean_14": [rolling_mean_14],
            "rolling_mean_28": [rolling_mean_28],
            "rolling_mean_56": [rolling_mean_56],

            "time_idx": [row["time_idx"]],
            "day_of_week": [row["day_of_week"]],
            "month": [row["month"]],
            "promotion_status": [row["promotion_status"]],
        })


        # -------------------------------------------------
        # 4. Match the categorical types used during training
        # -------------------------------------------------
        # We use the SAME category levels that X_final used.
        # This is important because LightGBM must interpret
        # categories consistently between training/prediction.
        for column in categorical_columns:
            future_features[column] = pd.Categorical(
                future_features[column],
                categories=X_final[column].cat.categories
            )


        # Keep the exact same feature order used during training.
        future_features = future_features[feature_columns]


        # -------------------------------------------------
        # 5. Predict this future day's demand
        # -------------------------------------------------
        prediction = model.predict(future_features)[0]

        # Demand cannot realistically be negative.
        prediction = max(0, prediction)


        # Save the prediction for our final result.
        predictions.append(prediction)


        # -------------------------------------------------
        # 6. IMPORTANT: add this prediction to history
        # -------------------------------------------------
        # Tomorrow's lag/rolling features can now use today's
        # predicted demand.
        sales_history.append(prediction)


    return predictions

In [ ]:
# =========================================================
# Generate forecasts for both promotion scenarios
# =========================================================

# Scenario A:
# Forecast assuming NO promotions during the next 14 days.
no_promotion_predictions = recursive_forecast(
    model=final_lightgbm_model,
    scenario_df=no_promotion_scenario,
    historical_sales=historical_sales_values
)


# Scenario B:
# Forecast using our planned promotion dates.
promotion_predictions = recursive_forecast(
    model=final_lightgbm_model,
    scenario_df=promotion_scenario,
    historical_sales=historical_sales_values
)


print("No-promotion predictions:")
print(no_promotion_predictions)

print("\nPromotion-scenario predictions:")
print(promotion_predictions)

In [ ]:
# =========================================================
# Compare the two promotion scenarios
# =========================================================

comparison_df = pd.DataFrame({
    "date": future_df["date"],

    # What the model predicts if we run no promotions.
    "no_promotion_forecast": no_promotion_predictions,

    # What the model predicts with our planned promotions.
    "promotion_forecast": promotion_predictions,

    # Keep the promotion plan visible in the final table.
    "promotion_status": promotion_scenario["promotion_status"],
})


# Calculate how much the forecast changed because
# we changed the future promotion plan.
comparison_df["forecast_difference"] = (
    comparison_df["promotion_forecast"]
    - comparison_df["no_promotion_forecast"]
)


# Round only for easier display.
display(
    comparison_df.round({
        "no_promotion_forecast": 2,
        "promotion_forecast": 2,
        "forecast_difference": 2
    })
)


# Compare total predicted demand across the entire 14-day period.
total_no_promotion = sum(no_promotion_predictions)
total_with_promotion = sum(promotion_predictions)

print(
    "Total forecast without promotions:",
    round(total_no_promotion, 2)
)

print(
    "Total forecast with promotions:",
    round(total_with_promotion, 2)
)

print(
    "Difference:",
    round(
        total_with_promotion - total_no_promotion,
        2
    )
)

In [ ]:
# =========================================================
# Diagnostic — historical demand by promotion status
# =========================================================

promotion_summary = (
    data.groupby("promotion_status", observed=True)["unit_sales"]
    .agg(
        count="count",
        mean="mean",
        median="median"
    )
    .round(2)
)

display(promotion_summary)

In [ ]:
# =========================================================
# Diagnostic — promotion behavior by weekday
# =========================================================

promotion_weekday_summary = (
    data.groupby(
        ["day_of_week", "promotion_status"],
        observed=True
    )["unit_sales"]
    .agg(
        count="count",
        mean="mean"
    )
    .round(2)
    .reset_index()
)

display(promotion_weekday_summary)

In [ ]:
# =========================================================
# Diagnostic — isolate the direct promotion effect
# =========================================================

# We will examine August 20 specifically.
test_date = pd.Timestamp("2017-08-20")


# ---------------------------------------------------------
# Reconstruct history up to August 19
# ---------------------------------------------------------

# Start with the real historical sales.
test_history = historical_sales_values.copy()


# Predict Aug 16 → Aug 19 using the NO-PROMOTION scenario.
# This gives us exactly the same history for both Aug 20 tests.
days_before_test = no_promotion_scenario[
    no_promotion_scenario["date"] < test_date
]

previous_predictions = recursive_forecast(
    model=final_lightgbm_model,
    scenario_df=days_before_test,
    historical_sales=historical_sales_values
)

test_history.extend(previous_predictions)


# ---------------------------------------------------------
# Build Aug 20 features from that SAME history
# ---------------------------------------------------------

aug20_row = no_promotion_scenario[
    no_promotion_scenario["date"] == test_date
].iloc[0]


base_features = {
    "lag_1": test_history[-1],
    "lag_7": test_history[-7],
    "lag_14": test_history[-14],
    "lag_28": test_history[-28],
    "lag_56": test_history[-56],

    "rolling_mean_7": np.mean(test_history[-7:]),
    "rolling_mean_14": np.mean(test_history[-14:]),
    "rolling_mean_28": np.mean(test_history[-28:]),
    "rolling_mean_56": np.mean(test_history[-56:]),

    "time_idx": aug20_row["time_idx"],
    "day_of_week": aug20_row["day_of_week"],
    "month": aug20_row["month"],
}


# Create TWO otherwise-identical rows.
direct_test = pd.DataFrame([
    {
        **base_features,
        "promotion_status": "no_promotion"
    },
    {
        **base_features,
        "promotion_status": "promotion"
    }
])


# Match categorical types used during model training.
for column in categorical_columns:
    direct_test[column] = pd.Categorical(
        direct_test[column],
        categories=X_final[column].cat.categories
    )


# Keep the exact training feature order.
direct_test = direct_test[feature_columns]


# Predict both rows.
direct_predictions = final_lightgbm_model.predict(
    direct_test
)


print(
    "Aug 20 prediction — NO promotion:",
    round(direct_predictions[0], 2)
)

print(
    "Aug 20 prediction — PROMOTION:",
    round(direct_predictions[1], 2)
)

print(
    "Direct promotion difference:",
    round(
        direct_predictions[1] - direct_predictions[0],
        2
    )
)

In [ ]:
# =========================================================
# Diagnostic 1 — Multi-row promotion perturbation
# =========================================================

# Start with historical LightGBM-ready rows.
#
# We use the same 13 features that were used to train
# the final LightGBM model.
perturbation_df = final_training_df[
    feature_columns
].copy()


# ---------------------------------------------------------
# Create TWO copies of every historical feature row.
#
# In one copy, we tell the model:
#     "Assume there is NO promotion."
#
# In the other copy:
#     "Assume there IS a promotion."
#
# Everything else remains exactly identical.
# ---------------------------------------------------------

no_promo_features = perturbation_df.copy()
promo_features = perturbation_df.copy()

no_promo_features["promotion_status"] = "no_promotion"
promo_features["promotion_status"] = "promotion"


# ---------------------------------------------------------
# Match the categorical representation used during training.
#
# LightGBM must see exactly the same category levels that
# it saw when final_lightgbm_model was trained.
# ---------------------------------------------------------

for column in categorical_columns:

    no_promo_features[column] = pd.Categorical(
        no_promo_features[column],
        categories=X_final[column].cat.categories
    )

    promo_features[column] = pd.Categorical(
        promo_features[column],
        categories=X_final[column].cat.categories
    )


# Keep exactly the same feature order as training.
no_promo_features = no_promo_features[feature_columns]
promo_features = promo_features[feature_columns]


print("Rows being tested:", len(perturbation_df))
print("No-promotion shape:", no_promo_features.shape)
print("Promotion shape:", promo_features.shape)

In [ ]:
# =========================================================
# Diagnostic 1 — Measure LightGBM's learned promotion effect
# =========================================================

# Predict every historical feature configuration while
# pretending there is NO promotion.
predictions_no_promo = final_lightgbm_model.predict(
    no_promo_features
)

# Predict those EXACT SAME feature configurations while
# pretending there IS a promotion.
predictions_promo = final_lightgbm_model.predict(
    promo_features
)


# Calculate the model's response to changing ONLY
# promotion_status.
promotion_effect = (
    predictions_promo
    - predictions_no_promo
)


# ---------------------------------------------------------
# Classify the model's response.
#
# Positive:
# promotion prediction > no-promotion prediction
#
# Negative:
# promotion prediction < no-promotion prediction
#
# Approximately zero:
# practically no change
# ---------------------------------------------------------

tolerance = 0.01

positive_count = np.sum(
    promotion_effect > tolerance
)

negative_count = np.sum(
    promotion_effect < -tolerance
)

neutral_count = np.sum(
    np.abs(promotion_effect) <= tolerance
)

total_rows = len(promotion_effect)


print("Total rows tested:", total_rows)

print(
    "Positive promotion response:",
    positive_count,
    f"({positive_count / total_rows * 100:.2f}%)"
)

print(
    "Negative promotion response:",
    negative_count,
    f"({negative_count / total_rows * 100:.2f}%)"
)

print(
    "Approximately neutral response:",
    neutral_count,
    f"({neutral_count / total_rows * 100:.2f}%)"
)

print(
    "\nAverage model promotion difference:",
    round(np.mean(promotion_effect), 2)
)

print(
    "Median model promotion difference:",
    round(np.median(promotion_effect), 2)
)

print(
    "Minimum difference:",
    round(np.min(promotion_effect), 2)
)

print(
    "Maximum difference:",
    round(np.max(promotion_effect), 2)
)

## Historical Promotion Adjustment

The LightGBM forecasting model retains `promotion_status` because rolling
backtesting showed that promotion information improves forecast accuracy.

However, directly changing the promotion feature and interpreting the
difference between predictions as the causal effect of a promotion is
not appropriate for observational sales data.

For business-facing promotion scenarios, we therefore estimate promotion
adjustments from comparable historical demand conditions. These estimates
represent historical associations rather than guaranteed causal effects.

In [ ]:
# =========================================================
# Step 1 — Create historical demand tiers
#
# We use the previous 28-day average demand to describe
# whether each historical day occurred during a relatively
# low, medium, or high-demand period.
# =========================================================

promotion_analysis_df = final_training_df.copy()

promotion_analysis_df["demand_tier"] = pd.qcut(
    promotion_analysis_df["rolling_mean_28"],
    q=3,
    labels=["low", "medium", "high"]
)

print("Historical rows:", len(promotion_analysis_df))

print("\nDemand tier counts:")
print(
    promotion_analysis_df["demand_tier"]
    .value_counts()
    .sort_index()
)

In [ ]:
# =========================================================
# Step 2 — Count comparable historical observations
#
# For each weekday and demand tier, count how many historical
# promotion and no-promotion observations are available.
#
# We inspect these counts before calculating promotion effects
# because very small groups would produce unreliable estimates.
# =========================================================

comparison_counts = (
    promotion_analysis_df[
        promotion_analysis_df["promotion_status"].isin(
            ["promotion", "no_promotion"]
        )
    ]
    .groupby(
        ["day_of_week", "demand_tier", "promotion_status"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(comparison_counts)

In [ ]:
# =========================================================
# Step 3 — Check which comparison groups have enough data
#
# We require at least 10 known promotion observations AND
# 10 known no-promotion observations before trusting a
# weekday + demand-tier comparison.
# =========================================================

MIN_GROUP_SIZE = 10

comparison_counts["enough_data"] = (
    (comparison_counts["promotion"] >= MIN_GROUP_SIZE)
    & (comparison_counts["no_promotion"] >= MIN_GROUP_SIZE)
)

display(comparison_counts)

In [ ]:
# =========================================================
# Step 4 — Calculate historical promotion adjustments
#
# First calculate the more specific:
#     weekday + demand tier
#
# We only keep groups with at least MIN_GROUP_SIZE
# observations on BOTH sides.
# =========================================================

specific_promotion_adjustments = []

for _, row in comparison_counts.iterrows():

    # Skip groups that do not have enough observations.
    if not row["enough_data"]:
        continue

    weekday = row["day_of_week"]
    demand_tier = row["demand_tier"]

    # Select historical observations belonging to this
    # exact weekday and demand environment.
    group_df = promotion_analysis_df[
        (promotion_analysis_df["day_of_week"] == weekday)
        & (promotion_analysis_df["demand_tier"] == demand_tier)
        & (
            promotion_analysis_df["promotion_status"].isin(
                ["promotion", "no_promotion"]
            )
        )
    ]

    # Calculate average observed sales separately for
    # promotional and non-promotional days.
    promo_mean = group_df.loc[
        group_df["promotion_status"] == "promotion",
        "unit_sales"
    ].mean()

    no_promo_mean = group_df.loc[
        group_df["promotion_status"] == "no_promotion",
        "unit_sales"
    ].mean()

    # Positive value means promotional days historically had
    # higher sales within this comparable group.
    adjustment = promo_mean - no_promo_mean

    specific_promotion_adjustments.append({
        "day_of_week": weekday,
        "demand_tier": demand_tier,
        "promo_mean_sales": promo_mean,
        "no_promo_mean_sales": no_promo_mean,
        "promotion_adjustment": adjustment,
    })


specific_adjustment_df = pd.DataFrame(
    specific_promotion_adjustments
)

display(specific_adjustment_df)

In [ ]:
# =========================================================
# Step 5 — Create fallback promotion adjustments
#
# If a weekday + demand-tier group does not contain enough
# observations, we fall back to the broader demand tier.
#
# Example:
#     Sunday + low has too little data
#              ↓
#     compare all low-demand promo vs no-promo days
# =========================================================

fallback_promotion_adjustments = []

# Only use rows where promotion status is actually known.
known_promotion_df = promotion_analysis_df[
    promotion_analysis_df["promotion_status"].isin(
        ["promotion", "no_promotion"]
    )
].copy()

# Calculate one fallback adjustment for each demand tier.
for demand_tier in ["low", "medium", "high"]:

    tier_df = known_promotion_df[
        known_promotion_df["demand_tier"] == demand_tier
    ]

    promo_sales = tier_df.loc[
        tier_df["promotion_status"] == "promotion",
        "unit_sales"
    ]

    no_promo_sales = tier_df.loc[
        tier_df["promotion_status"] == "no_promotion",
        "unit_sales"
    ]

    promo_mean = promo_sales.mean()
    no_promo_mean = no_promo_sales.mean()

    adjustment = promo_mean - no_promo_mean

    fallback_promotion_adjustments.append({
        "demand_tier": demand_tier,
        "promo_count": len(promo_sales),
        "no_promo_count": len(no_promo_sales),
        "promo_mean_sales": promo_mean,
        "no_promo_mean_sales": no_promo_mean,
        "promotion_adjustment": adjustment,
    })


fallback_adjustment_df = pd.DataFrame(
    fallback_promotion_adjustments
)

display(fallback_adjustment_df)

In [ ]:
# =========================================================
# Step 6 — Promotion adjustment lookup
#
# Try the more specific weekday + demand-tier estimate first.
# If that estimate is unavailable because of insufficient
# historical observations, use the broader demand-tier
# fallback estimate.
# =========================================================

def get_promotion_adjustment(day_of_week, demand_tier):

    # -----------------------------------------------------
    # 1. Look for a reliable weekday + demand-tier estimate.
    # -----------------------------------------------------
    specific_match = specific_adjustment_df[
        (specific_adjustment_df["day_of_week"] == day_of_week)
        & (specific_adjustment_df["demand_tier"] == demand_tier)
    ]

    if not specific_match.empty:

        adjustment = specific_match[
            "promotion_adjustment"
        ].iloc[0]

        return {
            "adjustment": adjustment,
            "source": "weekday_and_demand_tier"
        }

    # -----------------------------------------------------
    # 2. If the specific group was unavailable, use the
    #    broader demand-tier fallback.
    # -----------------------------------------------------
    fallback_match = fallback_adjustment_df[
        fallback_adjustment_df["demand_tier"] == demand_tier
    ]

    adjustment = fallback_match[
        "promotion_adjustment"
    ].iloc[0]

    return {
        "adjustment": adjustment,
        "source": "demand_tier_fallback"
    }


In [ ]:
# Test the lookup with several historical situations.

print(
    "Sunday + high:",
    get_promotion_adjustment("Sunday", "high")
)

print(
    "Sunday + low:",
    get_promotion_adjustment("Sunday", "low")
)

print(
    "Friday + high:",
    get_promotion_adjustment("Friday", "high")
)

In [ ]:
# =========================================================
# Step 7 — Save the historical demand-tier boundaries
#
# Historical demand tiers were created using rolling_mean_28.
# We now save the two cutoff values so that future dates can
# be assigned to low, medium, or high demand environments.
# =========================================================

low_cutoff = promotion_analysis_df.loc[
    promotion_analysis_df["demand_tier"] == "low",
    "rolling_mean_28"
].max()

medium_cutoff = promotion_analysis_df.loc[
    promotion_analysis_df["demand_tier"] == "medium",
    "rolling_mean_28"
].max()


def classify_demand_tier(rolling_mean_28):
    """
    Classify a future demand environment using the same
    historical boundaries used to create our demand tiers.
    """

    if rolling_mean_28 <= low_cutoff:
        return "low"

    elif rolling_mean_28 <= medium_cutoff:
        return "medium"

    else:
        return "high"


print("Low-demand upper cutoff:", low_cutoff)
print("Medium-demand upper cutoff:", medium_cutoff)

# Test that the classification function works.
print("\nExample classifications:")
print("4.0 ->", classify_demand_tier(4.0))
print("6.0 ->", classify_demand_tier(6.0))
print("9.0 ->", classify_demand_tier(9.0))

In [30]:
# =========================================================
# Step 8 — Forecast a promotion scenario using the
# historical promotion adjustment.
#
# LightGBM creates the baseline demand forecast while holding
# promotion_status at "no_promotion".
#
# If the user schedules a promotion, we estimate the
# additional demand using our historical comparison system.
# =========================================================

def forecast_with_historical_promotion_adjustment(
    model,
    historical_sales,
    future_calendar,
    feature_columns,
    categorical_columns,
    category_reference
):
    # Copy the historical sales because recursive forecasting
    # will append each new prediction to this list.
    history = list(historical_sales)

    results = []

    for _, future_row in future_calendar.iterrows():

        # -------------------------------------------------
        # Calculate the current demand environment BEFORE
        # predicting this future day.
        #
        # Because history contains only previous actual sales
        # and previous recursive predictions, this does not
        # use the future day's true sales.
        # -------------------------------------------------
        current_rolling_mean_28 = np.mean(history[-28:])

        demand_tier = classify_demand_tier(
            current_rolling_mean_28
        )

        # -------------------------------------------------
        # Create the LightGBM BASELINE prediction.
        #
        # Important:
        # We deliberately hold promotion_status at
        # "no_promotion" here. Promotion impact will be
        # handled separately below.
        # -------------------------------------------------
        prediction_row = pd.DataFrame({
            "lag_1": [history[-1]],
            "lag_7": [history[-7]],
            "lag_14": [history[-14]],
            "lag_28": [history[-28]],
            "lag_56": [history[-56]],

            "rolling_mean_7": [
                np.mean(history[-7:])
            ],
            "rolling_mean_14": [
                np.mean(history[-14:])
            ],
            "rolling_mean_28": [
                current_rolling_mean_28
            ],
            "rolling_mean_56": [
                np.mean(history[-56:])
            ],

            "time_idx": [future_row["time_idx"]],
            "day_of_week": [future_row["day_of_week"]],
            "month": [future_row["month"]],

            # Baseline = no promotion.
            "promotion_status": ["no_promotion"],
        })

        # Make categorical columns use the same categories
        # that the trained LightGBM model saw during training.
        for column in categorical_columns:
            prediction_row[column] = pd.Categorical(
                prediction_row[column],
                categories=category_reference[
                    column
                ].cat.categories
            )

        prediction_row = prediction_row[
            feature_columns
        ]

        # Predict baseline demand.
        baseline_prediction = model.predict(
            prediction_row
        )[0]

        baseline_prediction = max(
            0.0,
            baseline_prediction
        )

        # -------------------------------------------------
        # Apply historical promotion adjustment ONLY when
        # the future calendar says a promotion is planned.
        # -------------------------------------------------
        planned_promotion = (
            future_row["promotion_status"]
            == "promotion"
        )

        promotion_adjustment = 0.0
        adjustment_source = "no_promotion"

        if planned_promotion:

            adjustment_info = get_promotion_adjustment(
                future_row["day_of_week"],
                demand_tier
            )

            promotion_adjustment = float(
                adjustment_info["adjustment"]
            )

            adjustment_source = adjustment_info[
                "source"
            ]

        # Final scenario forecast.
        final_prediction = (
            baseline_prediction
            + promotion_adjustment
        )

        final_prediction = max(
            0.0,
            final_prediction
        )

        # IMPORTANT:
        # The final scenario prediction becomes part of the
        # history for forecasting the following day.
        history.append(final_prediction)

        results.append({
            "date": future_row["date"],
            "day_of_week": future_row["day_of_week"],
            "promotion_status": future_row[
                "promotion_status"
            ],
            "rolling_mean_28": current_rolling_mean_28,
            "demand_tier": demand_tier,
            "baseline_forecast": baseline_prediction,
            "promotion_adjustment": promotion_adjustment,
            "adjustment_source": adjustment_source,
            "final_forecast": final_prediction,
        })

    return pd.DataFrame(results)

In [32]:
# =========================================================
# Step 9 — Run the final 14-day promotion scenario
# =========================================================

promotion_scenario_results = (
    forecast_with_historical_promotion_adjustment(
        model=final_lightgbm_model,

        # Historical sales available before the forecast starts.
        historical_sales=historical_sales,

        # Our 14-day calendar containing the user's
        # planned promotion dates.
        future_calendar=promotion_scenario,

        # Use the same features used to train LightGBM.
        feature_columns=feature_columns,
        categorical_columns=categorical_columns,

        # X_final contains the categorical definitions
        # that LightGBM saw during training.
        category_reference=X_final,
    )
)

display(promotion_scenario_results)

C:\Users\karra\AppData\Local\Temp\ipykernel_9448\1838599139.py:81: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  prediction_row[column] = pd.Categorical(
C:\Users\karra\AppData\Local\Temp\ipykernel_9448\1838599139.py:81: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  prediction_row[column] = pd.Categorical(
C:\Users\karra\AppData\Local\Temp\ipykernel_9448\1838599139.py:81: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  prediction_row[column] = pd.Categorical(
C:\Users\karra\AppData\Local\Temp\ipykernel_9448\1838599139.py:81: Pandas4Warning: Constructing a Categorical with a dtype and values containi

,date,day_of_week,promotion_status,rolling_mean_28,demand_tier,baseline_forecast,promotion_adjustment,adjustment_source,final_forecast
0,2017-08-16,Wednesday,no_promotion,7.035714,high,4.410217,0.000000,no_promotion,4.410217
1,2017-08-17,Thursday,no_promotion,7.050365,high,5.347132,0.000000,no_promotion,5.347132
2,2017-08-18,Friday,no_promotion,7.134191,high,9.666766,0.000000,no_promotion,9.666766
3,2017-08-19,Saturday,no_promotion,7.229433,high,8.851894,0.000000,no_promotion,8.851894
4,2017-08-20,Sunday,promotion,7.152715,high,8.779484,2.557895,weekday_and_demand_tier,11.337379
5,2017-08-21,Monday,promotion,7.164764,high,6.023723,1.298246,weekday_and_demand_tier,7.321968
6,2017-08-22,Tuesday,no_promotion,7.033406,high,5.066959,0.000000,no_promotion,5.066959
7,2017-08-23,Wednesday,no_promotion,7.035797,high,5.055650,0.000000,no_promotion,5.055650
8,2017-08-24,Thursday,no_promotion,7.073499,high,5.893334,0.000000,no_promotion,5.893334
9,2017-08-25,Friday,no_promotion,7.105404,high,9.573292,0.000000,no_promotion,9.573292


In [34]:
# =========================================================
# Step 10 — Final promotion scenario sanity check
#
# Compare:
#   Scenario A → no promotions during the next 14 days
#   Scenario B → promotions on Aug 20, Aug 21, and Aug 27
#
# Both scenarios use the SAME final forecasting method.
# =========================================================

# Run Scenario A: no future promotions.
final_no_promotion_results = (
    forecast_with_historical_promotion_adjustment(
        model=final_lightgbm_model,
        historical_sales=historical_sales,
        future_calendar=no_promotion_scenario,
        feature_columns=feature_columns,
        categorical_columns=categorical_columns,
        category_reference=X_final,
    )
)


# Scenario B was already calculated in Step 9.
final_promotion_results = promotion_scenario_results


# ---------------------------------------------------------
# Calculate total forecast demand across the 14-day period.
# ---------------------------------------------------------

no_promotion_total = (
    final_no_promotion_results["final_forecast"].sum()
)

promotion_total = (
    final_promotion_results["final_forecast"].sum()
)

estimated_additional_demand = (
    promotion_total - no_promotion_total
)


# ---------------------------------------------------------
# Display the final comparison.
# ---------------------------------------------------------

print(
    "14-day forecast with NO promotions:",
    round(no_promotion_total, 2)
)

print(
    "14-day forecast with planned promotions:",
    round(promotion_total, 2)
)

print(
    "Estimated additional demand associated "
    "with planned promotions:",
    round(estimated_additional_demand, 2)
)

14-day forecast with NO promotions: 97.08
14-day forecast with planned promotions: 101.8
Estimated additional demand associated with planned promotions: 4.72


C:\Users\karra\AppData\Local\Temp\ipykernel_9448\1838599139.py:81: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  prediction_row[column] = pd.Categorical(
C:\Users\karra\AppData\Local\Temp\ipykernel_9448\1838599139.py:81: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  prediction_row[column] = pd.Categorical(
C:\Users\karra\AppData\Local\Temp\ipykernel_9448\1838599139.py:81: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  prediction_row[column] = pd.Categorical(
C:\Users\karra\AppData\Local\Temp\ipykernel_9448\1838599139.py:81: Pandas4Warning: Constructing a Categorical with a dtype and values containi